In [1]:
import pandas as pd
import numpy as np
from utils.metric import metric
from lifelines import KaplanMeierFitter
from lifelines import CoxPHFitter

In [2]:
train = pd.read_csv('../../data/train.csv')
val = pd.read_csv("../../data/val.csv")

In [3]:
train.head()

,event_id,num_perimeters_0_5h,dt_first_last_0_5h,low_temporal_resolution_0_5h,area_first_ha,area_growth_abs_0_5h,area_growth_rel_0_5h,area_growth_rate_ha_per_h,log1p_area_first,log1p_growth,...,dist_fit_r2_0_5h,alignment_cos,alignment_abs,cross_track_component,along_track_speed,event_start_hour,event_start_dayofweek,event_start_month,time_to_hit_hours,event
0,20307683,1,0.000000,1,37.726426,0.000000,0.000000,0.000000,3.656522,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,21,1,3,57.643807,0
1,60132804,13,4.819051,0,1630.389625,2508.041442,1.538308,520.443033,7.397187,7.827656,...,0.502981,0.902628,0.902628,-182.682530,383.099186,20,2,7,1.262348,1
2,90380661,1,0.000000,1,68.796478,0.000000,0.000000,0.000000,4.245584,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,15,2,8,46.530571,0
3,49907151,12,4.829815,0,200.386653,314.052044,1.567230,65.023617,5.305227,5.752738,...,0.638522,0.994594,0.994594,14.077093,134.831196,20,3,8,0.508562,1
4,18104294,1,0.000000,1,147.357925,0.000000,0.000000,0.000000,4.999628,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,14,1,9,23.843843,1


In [4]:
val.head()

,event_id,num_perimeters_0_5h,dt_first_last_0_5h,low_temporal_resolution_0_5h,area_first_ha,area_growth_abs_0_5h,area_growth_rel_0_5h,area_growth_rate_ha_per_h,log1p_area_first,log1p_growth,...,dist_fit_r2_0_5h,alignment_cos,alignment_abs,cross_track_component,along_track_speed,event_start_hour,event_start_dayofweek,event_start_month,time_to_hit_hours,event
0,61455506,1,0.000000,1,98.473215,0.000000,0.000000,0.000000,4.599888,0.000000,...,0.0,0.000000,0.000000,0.000000,0.000000,19,4,6,66.640791,0
1,68911231,2,2.278508,0,228.122411,0.000000,0.000000,0.000000,5.434256,0.000000,...,0.0,-0.663145,0.663145,0.000000,-0.000000,22,1,7,0.867177,1
2,49304487,1,0.000000,1,0.346854,0.000000,0.000000,0.000000,0.297772,0.000000,...,0.0,0.000000,0.000000,0.000000,0.000000,23,0,3,17.636732,0
3,80550714,17,3.164563,0,53.945875,280.295475,5.195865,88.573191,4.006349,5.639406,...,0.0,0.637211,0.637211,181.240473,149.850667,18,0,6,3.274951,1
4,18032557,1,0.000000,1,1749.278716,0.000000,0.000000,0.000000,7.467530,0.000000,...,0.0,0.000000,0.000000,0.000000,0.000000,20,2,6,64.567485,0


In [5]:
kmf = KaplanMeierFitter()
kmf.fit(durations=train["time_to_hit_hours"], event_observed=train["event"])

def km_probs_for_df(df):
    S12 = float(kmf.predict(12))
    S24 = float(kmf.predict(24))
    S48 = float(kmf.predict(48))
    S72 = float(kmf.predict(72))

    p12, p24, p48, p72 = (1-S12), (1-S24), (1-S48), (1-S72)

    probs = np.tile([p12, p24, p48, p72], (len(df), 1))
    return probs

val_probs = km_probs_for_df(val)

c, b, score = metric(
    val_probs,
    val["time_to_hit_hours"].values,
    val["event"].values
)

print("KM baseline")
print("C-index:", c)
print("Weighted Brier:", b)
print("Score:", score)

KM baseline
C-index: 0.5
Weighted Brier: 0.2835795448707349
Score: 0.6514943185904856


In [6]:
TIME_COL = "time_to_hit_hours"
EVENT_COL = "event"

ignore = {TIME_COL, EVENT_COL}
feature_cols = [c for c in train.columns if c not in ignore]

cox_train = train[feature_cols + [TIME_COL, EVENT_COL]].copy()

cph = CoxPHFitter(penalizer=1e-3)  
cph.fit(cox_train, duration_col=TIME_COL, event_col=EVENT_COL)

print(cph.summary.sort_values("p", ascending=True).head(15))

                                coef  exp(coef)      se(coef)  coef lower 95%  \
covariate                                                                       
dist_min_ci_0_5h       -4.609516e-05   0.999954  9.626693e-06   -6.496313e-05   
num_perimeters_0_5h     9.127503e-01   2.491165  2.462829e-01    4.300447e-01   
dt_first_last_0_5h     -4.318354e-01   0.649316  2.421755e-01   -9.064907e-01   
alignment_cos          -7.405851e-01   0.476835  4.392592e-01   -1.601517e+00   
cross_track_component  -2.082880e-02   0.979387  1.269213e-02   -4.570492e-02   
spread_bearing_sin     -1.694245e+00   0.183738  1.061591e+00   -3.774925e+00   
dist_fit_r2_0_5h        2.862114e+00  17.498479  1.891623e+00   -8.453997e-01   
spread_bearing_deg     -1.547816e-02   0.984641  1.061597e-02   -3.628508e-02   
alignment_abs           1.682171e+00   5.377218  1.160562e+00   -5.924889e-01   
event_start_month       1.866063e-01   1.205153  1.316578e-01   -7.143827e-02   
area_first_ha          -6.27

In [7]:
HORIZONS = [12, 24, 48, 72]

def cox_predict_probs(cph, df, feature_cols, horizons=HORIZONS):
    X = df[feature_cols]
    surv = cph.predict_survival_function(X, times=horizons) 
    S = surv.values.T  
    P = 1.0 - S 
    P = np.clip(P, 0.0, 1.0)
    P = np.maximum.accumulate(P, axis=1)
    return P  

val_probs = cox_predict_probs(cph, val, feature_cols)
c, b, score = metric(val_probs, val[TIME_COL].values, val[EVENT_COL].values)

print("CoxPH baseline")
print("C-index:", c)
print("Weighted Brier:", b)
print("Score:", score)

CoxPH baseline
C-index: 0.8275862068965517
Weighted Brier: 0.1265992475963002
Score: 0.8596563887515554
